# Archived Spark implementation
Historical source from the previous laptop. Run the notebooks one directory above for the current portable pipeline. Outputs have been cleared to avoid presenting old results as current.

# 01b - SPUD playlist preprocessing

This notebook independently audits and cleans the University of Glasgow SPUD playlist dataset. The source is a normalized SQLite database containing playlists, tracks, artists, albums, and playlist-track relationships with Spotify IDs.

The notebook uses Python's read-only SQLite driver only to stream the relational tables into partitioned CSV staging files. All quality checks, joins, transformations, statistics, and Parquet outputs are then performed with PySpark. No model is trained here.

## 1. Environment and paths

Run `scripts/setup_environment.ps1` first. The downloaded source must exist at `data/raw/playlists/spud/spud.sqlite`.

In [ ]:
import os
import sys
import csv
import json
import sqlite3
from pathlib import Path


def find_project_root():
    required = Path("data/raw/playlists/spud/spud.sqlite")
    anchors = [Path.cwd().resolve(), Path(sys.executable).resolve()]
    for anchor in anchors:
        for parent in [anchor, *anchor.parents]:
            for candidate in [parent, parent / "art_xharra"]:
                if (candidate / required).exists():
                    return candidate.resolve()
    raise FileNotFoundError(
        "Could not find data/raw/playlists/spud/spud.sqlite. "
        "Download/extract the SPUD archive before running this notebook."
    )


PROJECT_ROOT = find_project_root()
JDK_ROOT = PROJECT_ROOT / ".tools/jdk17/jdk-17.0.20+8"
HADOOP_ROOT = PROJECT_ROOT / ".tools/hadoop"
SQLITE_PATH = PROJECT_ROOT / "data/raw/playlists/spud/spud.sqlite"
STAGING_ROOT = PROJECT_ROOT / "data/interim/spud_csv"
OUTPUT_ROOT = PROJECT_ROOT / "data/processed/playlists"
PLAYLISTS_OUTPUT = OUTPUT_ROOT / "spud_playlists_clean.parquet"
TRACKS_OUTPUT = OUTPUT_ROOT / "spud_tracks_clean.parquet"
EDGES_OUTPUT = OUTPUT_ROOT / "spud_playlist_track_edges_clean.parquet"
REPORTS_ROOT = PROJECT_ROOT / "reports"

if not JDK_ROOT.exists():
    raise FileNotFoundError(f"Project Java runtime is missing: {JDK_ROOT}")
os.environ["JAVA_HOME"] = str(JDK_ROOT)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["PATH"] = str(JDK_ROOT / "bin") + os.pathsep + os.environ.get("PATH", "")
if os.name == "nt":
    if not (HADOOP_ROOT / "bin/winutils.exe").exists():
        raise FileNotFoundError(f"Windows Hadoop helper is missing: {HADOOP_ROOT / 'bin/winutils.exe'}")
    os.environ["HADOOP_HOME"] = str(HADOOP_ROOT)
    os.environ["PATH"] = str(HADOOP_ROOT / "bin") + os.pathsep + os.environ["PATH"]

from pyspark import StorageLevel
from pyspark.sql import SparkSession, functions as F, types as T
import pandas as pd

spark = (
    SparkSession.builder.master("local[*]")
    .appName("SPUD-Playlist-Preprocessing")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.execution.arrow.pyspark.enabled", "false")
    .config("spark.driver.memory", "4g")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
STAGING_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
REPORTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"SQLite source: {SQLITE_PATH} ({SQLITE_PATH.stat().st_size / 1024**2:.2f} MiB)")
print(f"Python       : {sys.version.split()[0]}")
print(f"Spark        : {spark.version}")

## 2. Inspect the SQLite source schema

The database is opened in read-only mode. Source table types are printed before any extraction so reviewers can see exactly what was supplied.

In [ ]:
connection = sqlite3.connect(f"file:{SQLITE_PATH}?mode=ro", uri=True)
source_tables = [
    row[0] for row in connection.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    )
]
sqlite_schema_rows = []
for table_name in source_tables:
    for column_id, column_name, declared_type, not_null, default_value, primary_key in connection.execute(
        f"PRAGMA table_info({table_name})"
    ):
        sqlite_schema_rows.append({
            "table": table_name, "column": column_name,
            "sqlite_type": declared_type or "untyped",
            "not_null": bool(not_null), "primary_key_position": primary_key,
        })
display(pd.DataFrame(sqlite_schema_rows))
connection.close()

## 3. Stream the required relational tables to partitioned staging CSV

Spark does not ship with an SQLite JDBC driver. Rather than introduce a hidden external dependency, this small extraction bridge uses the Python standard library and writes deterministic CSV parts. Rerunning the cell replaces only `data/interim/spud_csv`.

In [ ]:
def export_query(dataset_name, query, columns, chunk_size=200_000):
    destination = STAGING_ROOT / dataset_name
    destination.mkdir(parents=True, exist_ok=True)
    for old_part in destination.glob("part-*.csv"):
        old_part.unlink()

    connection = sqlite3.connect(f"file:{SQLITE_PATH}?mode=ro", uri=True)
    cursor = connection.execute(query)
    total_rows = 0
    part_number = 0
    try:
        while True:
            rows = cursor.fetchmany(chunk_size)
            if not rows:
                break
            part_path = destination / f"part-{part_number:05d}.csv"
            with part_path.open("w", encoding="utf-8", newline="") as output_file:
                writer = csv.writer(output_file, quoting=csv.QUOTE_MINIMAL)
                writer.writerow(columns)
                writer.writerows(rows)
            total_rows += len(rows)
            part_number += 1
    finally:
        connection.close()
    print(f"Exported {dataset_name}: {total_rows:,} rows in {part_number} part(s)")
    return total_rows

sqlite_export_counts = {}
sqlite_export_counts["playlists"] = export_query(
    "playlists",
    "SELECT playlistid, title, duration FROM lastfmplaylists ORDER BY playlistid",
    ["playlist_id", "playlist_title", "source_duration_seconds"],
)
sqlite_export_counts["tracks"] = export_query(
    "tracks",
    """
    SELECT t.trackid, t.spotifyid, t.title, t.artist, ar.spotifyid, ar.name,
           t.album, al.spotifyid, al.name, t.popularity, t.duration
    FROM tracks t
    LEFT JOIN artists ar ON t.artist = ar.artistid
    LEFT JOIN albums al ON t.album = al.albumid
    ORDER BY t.trackid
    """,
    [
        "spud_track_id", "spotify_track_id", "track_title",
        "spud_artist_id", "spotify_artist_id", "artist_name",
        "spud_album_id", "spotify_album_id", "album_name",
        "spud_popularity", "duration_seconds",
    ],
)
sqlite_export_counts["edges"] = export_query(
    "edges",
    "SELECT playlist, track FROM lastfmplayliststracks ORDER BY playlist, track",
    ["playlist_id", "spud_track_id"],
)

## 4. Read with explicit Spark schemas and print every type

Explicit schemas prevent automatic inference from silently changing an ID or duration type.

In [ ]:
PLAYLIST_SCHEMA = T.StructType([
    T.StructField("playlist_id", T.LongType(), True),
    T.StructField("playlist_title", T.StringType(), True),
    T.StructField("source_duration_seconds", T.DoubleType(), True),
])
TRACK_SCHEMA = T.StructType([
    T.StructField("spud_track_id", T.LongType(), True),
    T.StructField("spotify_track_id", T.StringType(), True),
    T.StructField("track_title", T.StringType(), True),
    T.StructField("spud_artist_id", T.LongType(), True),
    T.StructField("spotify_artist_id", T.StringType(), True),
    T.StructField("artist_name", T.StringType(), True),
    T.StructField("spud_album_id", T.LongType(), True),
    T.StructField("spotify_album_id", T.StringType(), True),
    T.StructField("album_name", T.StringType(), True),
    T.StructField("spud_popularity", T.DoubleType(), True),
    T.StructField("duration_seconds", T.DoubleType(), True),
])
EDGE_SCHEMA = T.StructType([
    T.StructField("playlist_id", T.LongType(), True),
    T.StructField("spud_track_id", T.LongType(), True),
])

def read_staged_csv(dataset_name, schema):
    return (
        spark.read.option("header", True).option("encoding", "UTF-8")
        .option("mode", "FAILFAST").option("multiLine", True)
        .option("quote", '"').option("escape", '"')
        .schema(schema).csv(str(STAGING_ROOT / dataset_name))
    )

raw_playlists = read_staged_csv("playlists", PLAYLIST_SCHEMA).persist(StorageLevel.MEMORY_AND_DISK)
raw_tracks = read_staged_csv("tracks", TRACK_SCHEMA).persist(StorageLevel.MEMORY_AND_DISK)
raw_edges = read_staged_csv("edges", EDGE_SCHEMA).persist(StorageLevel.MEMORY_AND_DISK)
raw_counts = {
    "playlists": raw_playlists.count(),
    "tracks": raw_tracks.count(),
    "edges": raw_edges.count(),
}
assert raw_counts == sqlite_export_counts

def spark_schema_table(dataset_name, dataframe):
    return pd.DataFrame([
        {"dataset": dataset_name, "column": field.name,
         "spark_type": field.dataType.simpleString(), "nullable": field.nullable}
        for field in dataframe.schema.fields
    ])

input_schema_pdf = pd.concat([
    spark_schema_table("raw_playlists", raw_playlists),
    spark_schema_table("raw_tracks", raw_tracks),
    spark_schema_table("raw_edges", raw_edges),
], ignore_index=True)
display(input_schema_pdf)
print(raw_counts)

## 5. Source-quality audit

Spotify IDs must be 22 alphanumeric characters. Popularity in SPUD is normalized to 0-0.99, unlike the 0-100 popularity scale in the first Spotify dataset. Track duration is measured in seconds.

In [ ]:
playlist_quality = raw_playlists.agg(
    F.sum((F.col("playlist_id").isNull()).cast("long")).alias("missing_id"),
    F.sum((F.col("playlist_title").isNull() | (F.length(F.trim("playlist_title")) == 0)).cast("long")).alias("missing_title"),
    F.sum((F.col("source_duration_seconds").isNull() | F.isnan("source_duration_seconds") | (F.col("source_duration_seconds") <= 0)).cast("long")).alias("invalid_source_duration"),
).first().asDict()
track_quality = raw_tracks.agg(
    F.sum((F.col("spud_track_id").isNull()).cast("long")).alias("missing_spud_track_id"),
    F.sum((F.col("spotify_track_id").isNull() | ~F.col("spotify_track_id").rlike(r"^[A-Za-z0-9]{22}$")).cast("long")).alias("invalid_spotify_track_id"),
    F.sum((F.col("spotify_artist_id").isNull() | ~F.col("spotify_artist_id").rlike(r"^[A-Za-z0-9]{22}$")).cast("long")).alias("invalid_spotify_artist_id"),
    F.sum((F.col("track_title").isNull() | (F.length(F.trim("track_title")) == 0)).cast("long")).alias("missing_track_title"),
    F.sum((F.col("duration_seconds").isNull() | F.isnan("duration_seconds") | (F.col("duration_seconds") <= 0)).cast("long")).alias("invalid_duration"),
    F.sum((F.col("spud_popularity").isNull() | F.isnan("spud_popularity") | ~F.col("spud_popularity").between(0, 1)).cast("long")).alias("invalid_popularity"),
).first().asDict()
duplicate_playlist_ids = raw_playlists.groupBy("playlist_id").count().where("count > 1").count()
duplicate_spotify_track_ids = raw_tracks.groupBy("spotify_track_id").count().where("count > 1").count()
duplicate_edge_rows = raw_edges.groupBy("playlist_id", "spud_track_id").count().where("count > 1").count()
orphan_playlist_edges = raw_edges.join(raw_playlists.select("playlist_id"), "playlist_id", "left_anti").count()
orphan_track_edges = raw_edges.join(raw_tracks.select("spud_track_id"), "spud_track_id", "left_anti").count()

quality_rows = [
    ("playlist", key, int(value)) for key, value in playlist_quality.items()
] + [
    ("track", key, int(value)) for key, value in track_quality.items()
] + [
    ("structural", "duplicate_playlist_ids", duplicate_playlist_ids),
    ("structural", "duplicate_spotify_track_ids", duplicate_spotify_track_ids),
    ("structural", "duplicate_edges", duplicate_edge_rows),
    ("structural", "orphan_playlist_edges", orphan_playlist_edges),
    ("structural", "orphan_track_edges", orphan_track_edges),
]
quality_pdf = pd.DataFrame(quality_rows, columns=["area", "check", "affected_rows"])
display(quality_pdf)

## 6. Clean tracks and playlist-track edges

Eleven nonpositive track durations are replaced with the median valid duration and marked with an indicator. No valid relationship is discarded for missing optional metadata.

In [ ]:
valid_duration_median = float(
    raw_tracks.where(F.col("duration_seconds") > 0).approxQuantile("duration_seconds", [0.5], 0.001)[0]
)

clean_tracks = raw_tracks
for string_column in [
    "spotify_track_id", "track_title", "spotify_artist_id",
    "artist_name", "spotify_album_id", "album_name",
]:
    clean_tracks = clean_tracks.withColumn(string_column, F.trim(F.col(string_column)))
clean_tracks = (
    clean_tracks
    .withColumn("duration_was_invalid",
        (F.col("duration_seconds").isNull() | F.isnan("duration_seconds") | (F.col("duration_seconds") <= 0)).cast("int"))
    .withColumn("duration_seconds",
        F.when(F.col("duration_was_invalid") == 1, F.lit(valid_duration_median)).otherwise(F.col("duration_seconds")))
    .where(F.col("spud_track_id").isNotNull())
    .where(F.col("spotify_track_id").rlike(r"^[A-Za-z0-9]{22}$"))
    .where(F.col("spotify_artist_id").rlike(r"^[A-Za-z0-9]{22}$"))
    .where(F.col("spud_popularity").between(0, 1))
    .dropDuplicates(["spotify_track_id"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
clean_track_count = clean_tracks.count()

clean_edges = (
    raw_edges.dropna(subset=["playlist_id", "spud_track_id"])
    .dropDuplicates(["playlist_id", "spud_track_id"])
    .join(clean_tracks.select("spud_track_id", "spotify_track_id"), "spud_track_id", "inner")
    .select("playlist_id", "spud_track_id", "spotify_track_id")
    .persist(StorageLevel.MEMORY_AND_DISK)
)
clean_edge_count = clean_edges.count()
print(f"Median valid track duration: {valid_duration_median:.3f} seconds")
print(f"Clean tracks: {clean_track_count:,} / {raw_counts['tracks']:,}")
print(f"Clean edges : {clean_edge_count:,} / {raw_counts['edges']:,}")

## 7. Recompute trustworthy playlist statistics and segments

The provided playlist-duration field has inconsistent outliers, so the reliable duration is recomputed as the sum of cleaned track durations. Playlists are not deleted: zero-edge and short playlists remain available for audit, while `eligible_for_link_prediction` identifies playlists with at least five tracks.

In [ ]:
playlist_aggregates = (
    clean_edges.join(
        clean_tracks.select(
            "spud_track_id", "duration_seconds", "spud_popularity", "spotify_artist_id"
        ),
        on="spud_track_id", how="inner",
    )
    .groupBy("playlist_id").agg(
        F.count("spud_track_id").alias("track_count"),
        F.countDistinct("spotify_artist_id").alias("artist_count"),
        F.sum("duration_seconds").alias("computed_duration_seconds"),
        F.avg("spud_popularity").alias("mean_spud_popularity"),
    )
)

clean_playlists = (
    raw_playlists
    .withColumn("title_was_missing",
        (F.col("playlist_title").isNull() | (F.length(F.trim("playlist_title")) == 0)).cast("int"))
    .withColumn("playlist_title",
        F.when(F.col("title_was_missing") == 1, F.lit("Untitled playlist"))
        .otherwise(F.trim(F.col("playlist_title"))))
    .join(playlist_aggregates, on="playlist_id", how="left")
    .fillna({"track_count": 0, "artist_count": 0, "computed_duration_seconds": 0.0})
    .withColumn("eligible_for_link_prediction", (F.col("track_count") >= 5).cast("int"))
    .withColumn("playlist_size_band_id",
        F.when(F.col("track_count") < 5, 0)
        .when(F.col("track_count") < 10, 1)
        .when(F.col("track_count") < 25, 2)
        .when(F.col("track_count") < 100, 3).otherwise(4))
    .withColumn("playlist_size_band",
        F.when(F.col("playlist_size_band_id") == 0, "under_5_not_eligible")
        .when(F.col("playlist_size_band_id") == 1, "small_5_9")
        .when(F.col("playlist_size_band_id") == 2, "medium_10_24")
        .when(F.col("playlist_size_band_id") == 3, "large_25_99").otherwise("very_large_100_plus"))
    .withColumn("title_character_count", F.length("playlist_title"))
    .withColumn("title_token_count", F.size(F.split(F.trim("playlist_title"), r"\s+")))
    .dropDuplicates(["playlist_id"])
    .persist(StorageLevel.MEMORY_AND_DISK)
)
clean_playlist_count = clean_playlists.count()
eligible_playlist_count = clean_playlists.where("eligible_for_link_prediction = 1").count()
zero_edge_playlist_count = clean_playlists.where("track_count = 0").count()

playlist_segments_pdf = (
    clean_playlists.groupBy("playlist_size_band_id", "playlist_size_band").count()
    .withColumn("percentage", F.round(100 * F.col("count") / F.lit(clean_playlist_count), 3))
    .orderBy("playlist_size_band_id").toPandas()
)
display(playlist_segments_pdf)
print(f"Eligible playlists (>=5 tracks): {eligible_playlist_count:,}")
print(f"Playlists without edges        : {zero_edge_playlist_count:,}")

## 8. Save and read back the clean Parquet datasets

In [ ]:
(clean_playlists.repartition(4).write.mode("overwrite").option("compression", "snappy").parquet(str(PLAYLISTS_OUTPUT)))
(clean_tracks.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(TRACKS_OUTPUT)))
(clean_edges.repartition(8).write.mode("overwrite").option("compression", "snappy").parquet(str(EDGES_OUTPUT)))

saved_playlists = spark.read.parquet(str(PLAYLISTS_OUTPUT))
saved_tracks = spark.read.parquet(str(TRACKS_OUTPUT))
saved_edges = spark.read.parquet(str(EDGES_OUTPUT))
saved_counts = {
    "playlists": saved_playlists.count(),
    "tracks": saved_tracks.count(),
    "edges": saved_edges.count(),
}
assert saved_counts == {
    "playlists": clean_playlist_count, "tracks": clean_track_count, "edges": clean_edge_count
}

output_schema_pdf = pd.concat([
    spark_schema_table("spud_playlists_clean", saved_playlists),
    spark_schema_table("spud_tracks_clean", saved_tracks),
    spark_schema_table("spud_playlist_track_edges_clean", saved_edges),
], ignore_index=True)
display(output_schema_pdf)
print("Saved outputs:")
for path, count in [
    (PLAYLISTS_OUTPUT, clean_playlist_count),
    (TRACKS_OUTPUT, clean_track_count),
    (EDGES_OUTPUT, clean_edge_count),
]:
    print(f"- {path.relative_to(PROJECT_ROOT)}: {count:,} rows")

## 9. Final validation and computed observations

The source does not record track order or addition timestamps for playlist edges. Consequently, the later dissertation evaluation can perform playlist-completion link prediction with random/stratified hidden edges, but it cannot claim next-track or temporal prediction.

In [ ]:
final_duplicate_edges = saved_edges.groupBy("playlist_id", "spotify_track_id").count().where("count > 1").count()
final_orphan_playlists = saved_edges.join(saved_playlists.select("playlist_id"), "playlist_id", "left_anti").count()
final_orphan_tracks = saved_edges.join(saved_tracks.select("spotify_track_id"), "spotify_track_id", "left_anti").count()
assert final_duplicate_edges == 0
assert final_orphan_playlists == 0
assert final_orphan_tracks == 0

quality_pdf.to_csv(REPORTS_ROOT / "spud_preprocessing_quality.csv", index=False)
playlist_segments_pdf.to_csv(REPORTS_ROOT / "spud_playlist_segments.csv", index=False)

print("OBSERVATIONS")
print("============")
print(f"1. The source contains {raw_counts['playlists']:,} playlists, {raw_counts['tracks']:,} tracks, and {raw_counts['edges']:,} playlist-track edges.")
print(f"2. All {clean_track_count:,} retained tracks have valid 22-character Spotify track and artist IDs.")
print(f"3. {track_quality['invalid_duration']:,} nonpositive track durations were replaced with the valid median ({valid_duration_median:.3f} seconds) and flagged.")
print(f"4. One missing playlist title was replaced with 'Untitled playlist' and marked by title_was_missing.")
print("5. Playlist duration and size were recomputed from cleaned edges because the supplied duration field contains inconsistent outliers.")
print(f"6. {zero_edge_playlist_count:,} zero-edge playlists were retained for audit but will not become graph nodes.")
print(f"7. {eligible_playlist_count:,} playlists contain at least five tracks and are eligible for seed/hidden-track evaluation.")
print(f"8. The cleaned relationship table has {clean_edge_count:,} unique edges with zero orphan endpoints.")
print("9. SPUD popularity uses a 0-1 scale; it must not be confused with the 0-100 popularity field in the original track dataset.")
print("10. Playlist edges have no order or timestamp, so evaluation will measure set-based playlist completion rather than next-track prediction.")